# 使用 Unsloth 对 DeepSeek-R1-Distill-Qwen-1.5B 模型进行 LoRA 微调

本 Notebook 展示了如何使用 `unsloth` 库对 `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` 模型进行高效的 QLoRA (Low-Rank Adaptation) 微调。

整个流程包括：
1.  环境准备与库导入
2.  加载预训练模型和分词器 (Tokenizer)。
3.  在微调前，对模型进行简单的推理测试。
4.  下载和格式化训练数据集
5.  使用 `unsloth` 的 `FastLanguageModel` 来为模型添加 LoRA 适配器。
6.  配置 `SFTTrainer` 监督微调训练配置。
7.  启动训练，并观察 Loss 变化情况
8.  保存微调后的模型
9.  测试训练后的生成结果

### 1. 环境准备与库导入

首先，我们需要安装并导入所有必要的库。`transformers` 用于加载模型和分词器，`unsloth` 用于高效微调，`trl` 提供了 `SFTTrainer`，而 `datasets` 用于处理数据。

**注意**: 在运行此 Notebook 之前，请确保已安装所有依赖包：
```bash
pip install -r requirements.txt
```

In [1]:
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig, DataCollatorForSeq2Seq
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 2. 加载预训练模型和分词器 (Tokenizer)

In [2]:
# 定义模型和一些基本参数
max_seq_length = 8192
dtype = None # None 表示自动选择 (Float16 a T4, V100, BFloat16 a Ampere)
load_in_4bit = True # 使用 4bit 量化加载

# 这是您的模型标识符，请替换为您正在使用的模型
# 例如："qwen-1.5b_lora_model"
# model_name = "qwen-1.5b_lora_model" 
# model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B" 
model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" 

# 这一步会返回一个经过 Unsloth 优化的模型和一个分词器
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


==((====))==  Unsloth 2025.8.5: Fast Qwen2 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 15.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### 3. 微调前推理测试

在对模型进行任何修改之前，我们先用它来生成一段文本，看看原始模型的表现如何。这可以作为我们微调效果的基准参考。

In [11]:
# 模型推理的 Prompt 模板
inference_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>{}
"""

In [4]:
FastLanguageModel.for_inference(model)

question = "男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。"

inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")
attention_mask = inputs.input_ids.ne(tokenizer.pad_token_id).long().to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)

In [5]:
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [6]:
response

['以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。\n请撰写一份恰当的回复，以完成该请求。\n在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。\n\n\n### Instruction:\n你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。\n请回答以下医学问题。\n\n### Question:\n男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。\n\n### Response:\n<think>\n好的，我现在需要仔细分析这位28岁的男性患者的情况，并进行初步的临床推理和诊断。首先，患者的症状包括头晕、脖子疼，偶尔还恶心。这些都是常见的神经系统症状，可能与脑力劳动有关，也有可能是长期使用某种药物或生活方式问题引起的。\n\n首先，我应该考虑是否可能有脑力损伤，比如脑力运动损伤（CVD）。这是一种常见但轻微的神经系统疾病，通常在工作压力或长期使用电子设备中常见。患者每天工作到半夜，这可能是一个长期的脑力劳动，也可能与长时间使用电脑、手机等设备相关。脑力运动损伤通常表现为头晕、恶心、呕吐等，这些症状与患者描述的情况相吻合。\n\n接下来，我需要评估是否存在其他可能的神经系统疾病，比如多发性硬化症（MS）或帕金森病（ dyst普罗尼亚）。这些疾病通常是由于长期的神经系统损伤或慢性疾病引起的。然而，患者的症状较为轻微，且没有明显的持续性，因此这些可能性可能较低。\n\n另外，患者是否可能有其他神经系统疾病，比如帕金森病或其他运动性神经系统疾病？帕金森病通常伴随运动性疼痛，而患者并没有描述这种症状，所以这可能不是他的主要问题。\n\n然后，我需要考虑是否存在脑力损伤或运动性神经系统疾病，如多发性硬化症。这些疾病通常有轻微的神经系统症状，如头晕、恶心，但患者没有持续的或严重的症状。因此，这可能不是患者的主诉。\n\n此外，我需要考虑是否存在其他神经系统疾病，如多发性硬化症或其他慢性神经系统疾病。这些疾病通常伴随轻微的神经系统症状，因此可能与患者的情况相符合。\n\n现在，我需要考虑患者的年龄28岁，这可能与脑力劳动相关，因为通常脑力劳动是20岁以上人群常见的。此外，患者每天工作到半夜，这可能是一个长期的脑力劳动，也有可能与长期使用某种设备或生活方式有关。\n\n接下来，我需

In [ ]:
print(response[0].split("### Response:")[1])

---

### 4. 下载和格式化训练数据集


医学推理数据集：https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/viewer/zh

![dataset](images/dataset.png)

In [3]:
# 模型训练的 Prompt 模板
train_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [ ]:
!hf auth login

In [4]:
EOS_TOKEN = tokenizer.eos_token # 添加 EOS Token

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        # 将 EOS Token 添加到样本最后
        text = train_prompt.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset_id = 'FreedomIntelligence/medical-o1-reasoning-SFT'
# dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "zh", split = "train")
dataset = load_dataset('json', data_files='dataset/medical_o1_sft_Chinese.json',  split = "train")

dataset = dataset.map(formatting_prompts_func, batched = True,)

In [14]:
dataset[0]["text"]

'以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。\n请撰写一份恰当的回复，以完成该请求。\n在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。\n\n\n### Instruction:\n你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。\n请回答以下医学问题。\n\n### Question:\n根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？\n\n### Response:\n<think>\n这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。\n\n用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。\n\n但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。\n\n等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。\n\n仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。\n\n回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？\n\n再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。\n\n看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。\n\n不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。\n\n

In [15]:
from IPython.display import display, Markdown

display(Markdown(dataset[0]["text"]))

以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？

### Response:
<think>
这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。

用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。

但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。

等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。

仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。

回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？

再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。

看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。

不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。

哦，该不会是夏季湿热，导致湿毒入侵，孩子的体质不能御，其病情发展成这样的感染？综合分析后我觉得‘蝼蛄疖’这个病名真是相当符合。
</think>
从中医的角度来看，你所描述的症状符合“蝼蛄疖”的病症。这种病症通常发生在头皮，表现为多处结节，溃破流脓，形成空洞，患处皮肤增厚且长期不愈合。湿热较重的夏季更容易导致这种病症的发展，特别是在免疫力较弱的儿童身上。建议结合中医的清热解毒、祛湿消肿的治疗方法进行处理，并配合专业的医疗建议进行详细诊断和治疗。
<｜end▁of▁sentence｜>

### 5. 使用 Unsloth 添加 LoRA 适配器

这是使用 `unsloth` 的核心步骤。我们调用 `FastLanguageModel.get_peft_model`，它会非常高效地为模型注入 LoRA 模块。

- `r`: LoRA 的秩 (rank)，是控制模型复杂度和参数量的关键超参数。
- `target_modules`: 指定要在哪些线性层（如注意力机制的 q, k, v, o 投影层）上应用 LoRA。
- `lora_alpha`: LoRA 的缩放因子，通常设置为 `r` 的两倍或与 `r` 相同。
- `use_gradient_checkpointing`: 一种节省显存的技术，对于训练大模型至关重要。

In [5]:
# 因为 `model` 对象现在是由 Unsloth 创建的，它包含了所有必需的属性
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=1432,
    use_rslora=False,
    loftq_config=None,
)
# 检查模型结构，确认 LoRA 适配器已添加
print(model)

Unsloth 2025.8.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

### 6. 配置 SFTTrainer

`SFTTrainer` (Supervised Fine-tuning Trainer) 是一个专门用于指令微调的训练器。我们需要配置 `TrainingArguments` 来指定所有的训练参数，如批量大小、学习率、优化器等。

In [6]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 1432,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

### 7. 开始训练

一切准备就绪后，调用 `trainer.train()` 即可开始微调过程。训练结束后，会返回包含训练统计信息（如训练损失）的对象。

In [7]:
trainer_stats = trainer.train()

# 打印训练统计信息
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,171 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.213700
2,3.257100
3,3.199300
4,3.196000
5,3.161800
6,2.961900
7,2.975300
8,2.931000
9,2.900400
10,2.769100


TrainOutput(global_step=60, training_loss=2.4800588766733807, metrics={'train_runtime': 274.9087, 'train_samples_per_second': 3.492, 'train_steps_per_second': 0.218, 'total_flos': 7418600951439360.0, 'train_loss': 2.4800588766733807})


### 8. 保存微调后的模型（Lora）

训练完成后，您可以再次进行推理，比较微调后的模型与原始模型的差异。如果对结果满意，可以使用 `model.save_pretrained("your_lora_adapter_path")` 来保存训练好的 LoRA 适配器。

In [8]:
model.save_pretrained("qwen-1.5b_lora_model")

In [9]:
tokenizer.save_pretrained("qwen-1.5b_lora_model")

('qwen-1.5b_lora_model/tokenizer_config.json',
 'qwen-1.5b_lora_model/special_tokens_map.json',
 'qwen-1.5b_lora_model/chat_template.jinja',
 'qwen-1.5b_lora_model/tokenizer.json')

In [20]:
# 模型保存方式二选一（要么使用上面的分开保存，要么使用这里的合并 Lora 保存）
# model.save_pretrained_merged("qwen-1.5b_lora_model", tokenizer, save_method="merged_16bit")

### 9. 测试训练后的生成结果

In [14]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

question="一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？", # Question
inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1000,
    use_cache=True,
)

In [15]:
output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(output[0].split("### Response:")[1])


<think>
嗯，病人已经得了急性阑尾炎，已经发5天了。他们开始感到腹痛，不过感觉好像很轻，稍微减轻了一点，但还是有点发热。他们在体检的时候发现右下腹有压痛的包块，这种包块通常是在阑尾炎里会有的，对吧？那这个时候，病人应该怎么办呢？

首先，我得考虑一下，这个包块可能是什么意思。是不是阑尾炎的结节呢？或者是别的什么包块？如果是阑尾炎，那可能需要一些药物来处理，比如镇痛药或者抗生素之类的。但如果是其他包块呢，比如肿瘤，那处理起来可能更复杂一些，需要检查一下肿瘤的大小和位置。

再想想，病人现在有腹痛，发热，这可能意味着他们的炎症状态比较严重。因为急性阑尾炎通常和炎症有关，所以这个病人的炎症可能比较严重。所以，他们得注意及时处理。

然后，我得考虑一下他们是否需要手术。如果是阑尾炎，手术可能是个好选择，因为阑尾炎处理起来比较复杂，而且手术后通常会恢复良好，不容易复发。但如果是其他包块呢，可能需要进一步检查，比如是否有肿瘤，或者是否需要手术。

还有，病人现在有腹痛，发热，这可能意味着他们的炎症状态比较严重，需要及时处理。如果他们需要手术的话，可能需要考虑一下术前的检查，比如血常规或者影像学检查，以确保他们的情况是正确的。

另外，病人有右下腹有压痛的包块，这可能意味着他们正在治疗阑尾炎。但如果是其他包块呢，可能需要考虑一下，比如是否需要手术或者其他处理方法。如果他们需要手术，可能需要考虑一下术前的检查，以确保他们的情况是正确的。

总的来说，病人现在有腹痛，发热，需要考虑是否需要手术，但如果是其他包块的话，可能需要进一步检查。所以，他们应该尽快检查，看是阑尾炎还是其他包块，然后根据情况采取相应的处理措施。

总之，病人现在需要尽快处理，可能需要考虑手术，但如果是其他包块的话，可能需要进一步检查，以确保处理是正确的。
</think>
病人已经患有急性阑尾炎，已经发5天，腹痛轻微但仍有发热。在体检时发现右下腹有压痛的包块。这种情况需要谨慎处理，首先需要考虑是阑尾炎还是其他包块。

急性阑尾炎通常伴随炎症，因此患者可能需要及时处理。如果发现压痛包块，尤其是与阑尾炎相关的，建议尽快考虑手术。手术通常可以缓解炎症并改善治疗效果，但需要在术前进行详细检查以确保处理是正确的。

然而，如果包块不是阑尾炎的结节，而可能是其他类型的包块，例如肿瘤，还需要进一步检查。如果有肿瘤，可能

In [16]:
def generate_response(question: str, model, tokenizer, inference_prompt: str, max_new_tokens: int = 1024) -> str:
    """
    使用指定的模型和分词器为给定的医学问题生成响应。

    Args:
        question (str): 需要模型回答的医学问题。
        model: 已加载的 Unsloth/Hugging Face 模型。
        tokenizer: 对应的分词器。
        inference_prompt (str): 用于格式化输入的 f-string 模板。
        max_new_tokens (int, optional): 生成响应的最大 token 数量。默认为 1024。

    Returns:
        str: 模型生成的响应文本，已去除 prompt 部分。
    """
    # 1. 使用模板格式化输入
    prompt = inference_prompt.format(
        question, # 填充问题
        "",       # 留空，让模型生成 CoT 和 Response
    )

    # 2. 将格式化后的 prompt 进行分词，并转移到 GPU
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    # 3. 使用模型生成输出
    # use_cache=True 用于加速解码过程
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )
    
    # 4. 将生成的 token 解码为文本
    # skip_special_tokens=True 会移除像 EOS_TOKEN 这样的特殊标记
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # 5. 切分字符串，只返回 "### Response:" 之后的部分
    # 使用 .split() 分割并获取响应内容，.strip() 用于去除可能存在的前后空白字符
    response_part = decoded_output.split("### Response:")
    if len(response_part) > 1:
        return response_part[1].strip()
    else:
        # 如果模型没有生成 "### Response:" 标记，则返回整个生成内容以供调试
        return decoded_output

In [18]:
my_question = "对于一名60岁男性患者，出现右侧胸疼并在X线检查中显示右侧肋膈角消失，诊断为肺结核伴右侧胸腔积液，请问哪一项实验室检查对了解胸水的性质更有帮助？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这位60岁的男孩，有右胸痛，X线检查显示右肋膈角消失，这似乎提示他可能有肺结核。肺结核通常会导致肺部积液，尤其是胸腔积液。现在，他得出了这个结论。

为了了解这个胸腔积液的具体性质，最直接的方法是做CT或MRI检查。这些检查可以提供详细的信息，比如细胞学检查可以发现肺结核病灶的具体结构，而CT或MRI还能看到积液的形态和大小，帮助判断是气体积液还是液体积液，以及这些积液的大小。

所以，CT或MRI检查对于了解胸腔积液的性质是很有用的。CT或MRI不仅能够提供详细的影像，还能帮助诊断出肺结核的病灶，同时也能给出肺结核病灶的具体结构，这样就能更准确地判断是气体还是液体积液。所以，CT或MRI检查对了解胸腔积液的性质是有帮助的。

嗯，看来CT或MRI检查确实是一个不错的选择。这能帮助我们更清楚地了解这个患者的肺部问题。
</think>
CT或MRI检查对于了解肺部积液的性质是有帮助的。这些检查可以提供详细的信息，比如细胞学检查可以发现肺结核病灶的具体结构，而CT或MRI还能看到积液的形态和大小，帮助判断是气体积液还是液体积液。此外，CT或MRI检查还能够帮助诊断出肺结核的病灶，从而更准确地判断是气体还是液体积液。


In [19]:
my_question = "对于一名 28 岁的男性患者，工作是程序员，常年熬夜，最近突然感觉头晕目眩，甚至有点恶心。请问有可能是什么疾病？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
嗯，这位28岁的男性患者最近感觉头晕目眩，还有点恶心，这看起来有点不寻常啊。我得仔细想想，这是什么病呢？

首先，他工作是程序员，长期熬夜，这可能和某些神经功能异常有关吧。再想想，头晕目眩，还有恶心，这让我想到脑部的某些异常，比如脑膜炎或者脑膜炎引起的脑膜炎。

脑膜炎有时候会引发一些症状，比如头晕、恶心、恶心、头痛、视力模糊、视力问题、视力模糊、视力问题，这都挺符合的。嗯，再想想，他是不是有其他可能？比如脑膜炎，或者脑膜炎引起的脑膜炎。

再想想，脑膜炎是不是一种很常见的情况？有时候患者可能在感染或者有其他感染的情况下，才会出现这些症状。嗯，再看看他是不是有其他可能，比如脑膜炎引起的脑膜炎。

哦，对了，脑膜炎引起的脑膜炎有时候还会有脑膜炎引起的脑膜炎，而这个脑膜炎可能在脑膜炎后有更严重的症状。嗯，这听起来有点道理。

所以，综合来看，这位28岁的男性患者可能得脑膜炎引起的脑膜炎，因为脑膜炎引起的脑膜炎会导致头晕目眩，还有恶心，这和患者的情况相符。

嗯，看来脑膜炎引起的脑膜炎是可能的原因之一。
</think>
这位28岁的男性患者最近有头晕目眩和恶心的症状，这可能与脑膜炎有关。脑膜炎是一种常见的神经系统疾病，通常会导致头晕、恶心、头痛、视力模糊等症状。脑膜炎引起的脑膜炎会导致更严重的症状，如脑膜炎引起的脑膜炎。因此，这位患者可能得脑膜炎引起的脑膜炎。


In [24]:
my_question = "对于一名 42 岁的男性患者，工作是程序员，常年熬夜，尿酸水平较高约450，最近右膝关节外侧痛，肿胀，弯曲困难。请问有可能是什么疾病？"

response = generate_response(my_question, model, tokenizer, inference_prompt, max_new_tokens=2048)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
嗯，这位42岁的男性患者工作是程序员，工作很累，经常熬夜，尿酸水平也挺高的，达到450。最近右膝关节外侧有痛，肿胀，弯曲困难。这让我想到一些可能的疾病。

首先，右膝关节的疼痛和肿胀，弯曲困难，可能跟膝关节的活动有关。嗯，这个情况让我想到，膝关节的肌肉可能有问题，特别是如果疼痛持续，肿胀和弯曲困难，可能跟膝关节的运动有关。

然后，尿酸水平高，这个数字是450，也挺高的。尿酸高通常是尿路问题，比如尿路感染或者尿道狭窄。不过，尿酸高还可能和某些感染有关，比如尿道炎。

另外，这个患者工作很累，经常熬夜，说明他的身体可能需要更多的营养和能量，这可能和营养不良有关。比如，缺乏某些营养素可能导致尿酸升高，或者因为营养不良导致的尿路感染。

还有，尿酸高还可能和某些感染有关，比如尿道炎。不过，尿酸高和尿道炎之间可能有些关联，但不一定完全相同。

再想想，这个患者的疼痛和肿胀，弯曲困难，可能与膝关节的运动有关，尤其是膝关节的活动。这可能与膝关节的肌肉功能有关，比如膝关节的肌肉是冗余的，可能需要更多的营养和能量支持。

如果这些症状和尿酸水平高结合起来，可能和尿道感染有关，因为尿道感染常常伴随着尿酸升高。或者，可能是因为尿道感染导致尿酸升高。

或者，考虑到尿酸高和尿道感染可能同时存在，这可能是因为尿道感染导致尿酸升高。不过，尿酸高还可能与某些感染有关，比如尿道炎。

另外，这个患者的疼痛和肿胀，弯曲困难，可能与膝关节的肌肉紧张有关，这可能与尿酸高有关，因为尿酸高通常和肌肉疲劳和酸性环境有关。

嗯，这些症状和尿酸高结合起来，可能与尿道感染有关，因为尿道感染通常伴随着尿酸升高。或者，可能是因为尿道感染导致尿酸升高。

或者，考虑尿酸高和尿道感染的关联，可能是因为尿道感染导致尿酸升高。或者，尿酸高和尿道炎的关联，也可能是因为尿道炎导致尿酸升高。

这可能和尿道炎有关，因为尿道炎通常伴随着尿酸升高。或者，尿酸高和尿道炎的关联。

嗯，可能是因为尿道炎导致尿酸升高，或者尿道炎和尿酸升高有关。

或者，考虑到尿酸高和尿道炎的关联，可能是因为尿道炎导致尿酸升高。

或者，尿酸高和尿道炎的关联，可能是因为尿道炎导致尿酸升高。

或者，尿酸高和尿道炎的关联，可能是因为尿道炎导致尿酸升高

In [25]:
my_question = "对于一名 42 岁的男性患者，工作是程序员，常年熬夜，尿酸水平较高约450，最近右膝关节外侧痛，肿胀，弯曲困难。请问有可能是什么疾病？"

response = generate_response(my_question, model, tokenizer, inference_prompt, max_new_tokens=2048)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这个42岁的男性患者工作是程序员，长期熬夜，尿酸水平也很高，这可能说明他的肾脏有问题。他最近右膝关节外侧疼痛，肿胀，弯曲困难，这让我想到可能是尿崩症，也就是尿崩症综合征。尿崩症通常是因为肾脏功能失调引起的。

尿崩症可以分为两种类型：一种是由于肾脏功能异常引起的，比如肾小球滤过性疾病的症状；另一种是由于肾脏功能异常导致的，比如肾小球重吸收性疾病的症状。这两种情况都需要通过尿常规来检查。

患者在检查中发现尿酸水平在正常范围内，但尿液中尿崩症的指标显示尿崩症存在。这表明他的肾脏功能确实有问题，导致尿崩症的发生。

为了进一步确认，需要做尿常规检查，以确定尿崩症的具体原因。如果发现尿崩症是由于肾脏功能异常引起的，那可能需要进行尿镜检查以排除肾小球滤过性疾病的可能性。

此外，患者最近工作很累，长期熬夜，尿酸水平高，这可能与尿崩症的病因有关，比如肾小球滤过性疾病的长期积累。

总之，结合患者的症状和尿常规检查结果，尿崩症综合征是合理的解释。
</think>
根据患者的症状和尿常规检查结果，可以考虑尿崩症综合征。尿崩症通常是因为肾脏功能异常引起的，其常见原因是肾小球滤过性疾病的症状。患者在检查中发现尿崩症，这表明肾脏可能在处理尿液时出现了异常。为了进一步确认，尿常规检查是必要的，以明确具体病因。如果尿崩症是由于肾脏功能异常引起的，建议进行尿镜检查来排除肾小球滤过性疾病的可能。此外，长期的熬夜和高尿酸水平也可能是尿崩症的一种潜在原因。
